# Practice Lab: Cost Engineering in Python (Lesson 7.3)

Module 7 · 8 exercises · Token counting + routing + caching + optimization

Exercises 1, 2, 5, 6, 7, 8 run locally. Ex 1, 2, 5 need `pip install tiktoken`.


# Lesson 7.3: Cost Engineering in Python

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 5, 6, 7, 8 run **locally**. Ex 1,2,5 need `pip install tiktoken`.


In [ ]:
# pip install tiktoken  (for production token counting)
import json, re, random
from collections import defaultdict




---
## Exercise 1 (Easy): Token Counting & Cost Calculation


In [ ]:
# YOUR CODE
pass




<details><summary>Solution</summary>


In [ ]:
# Token estimator (~4 chars/token, matches tiktoken cl100k_base avg)
# In production: pip install tiktoken; enc = tiktoken.get_encoding('cl100k_base')
def ct(text):
    return max(1, len(text.split()) * 13 // 10)  # ~1.3 tokens/word

PRICING = {
    "GPT-4o":       {"input": 2.50,  "output": 10.00},
    "GPT-4o-mini":  {"input": 0.15,  "output": 0.60},
    "DeepSeek V3":  {"input": 0.27,  "output": 1.10},
    "Gemini Flash": {"input": 0.075, "output": 0.30},
}

prompts = [
    ("Simple greeting", "Hello, how are you?"),
    ("Short question", "What is Python?"),
    ("Medium query", "Explain REST vs GraphQL APIs in detail."),
    ("Long instruction", "You are an expert. Analyze this dataset. " * 10),
    ("Code request", "Write a Python binary search with tests. " * 3),
    ("System prompt", "You are a helpful AI for Netsetos edtech. " * 8),
    ("Translation", "Translate to Hindi: " + "The quick brown fox. " * 5),
    ("Summarization", "Summarize: " + "Global temperatures rose. " * 15),
    ("Multi-turn", ("User: What is ML?\nAssistant: ML is...\n") * 5),
    ("RAG context", "Documents:\n" + "Doc: AI in healthcare.\n" * 10),
]
output_tokens = [15, 30, 120, 200, 250, 50, 180, 150, 200, 300]

print(f"{'#':<4} {'Prompt':<20} {'In':<7} {'Out':<7} "
      f"{'GPT-4o $':<10} {'Flash $':<10} {'Ratio'}")
print("=" * 72)

totals = {m: 0 for m in PRICING}
most_exp = (0, "", 0)

for i, ((name, text), out) in enumerate(zip(prompts, output_tokens)):
    inp = ct(text)
    costs = {}
    for model, p in PRICING.items():
        c = (inp * p["input"] + out * p["output"]) / 1e6
        costs[model] = c
        totals[model] += c
    ratio = costs["GPT-4o"] / costs["Gemini Flash"]
    if costs["GPT-4o"] > most_exp[2]:
        most_exp = (i+1, name, costs["GPT-4o"])
    print(f"{i+1:<4} {name:<20} {inp:<7} {out:<7} "
          f"${costs['GPT-4o']:<9.6f} ${costs['Gemini Flash']:<9.6f} "
          f"{ratio:.0f}x")

print(f"\nMost expensive: #{most_exp[0]} '{most_exp[1]}'")
print(f"\nMonthly (10K/day):")
print(f"{'Model':<15} {'Monthly $':<10} {'Monthly INR'}")
for model, total in totals.items():
    monthly = total / len(prompts) * 300_000
    print(f"{model:<15} ${monthly:<9.2f} Rs {monthly*84:>8,.0f}")




</details>


---
## Exercise 2 (Easy): System Prompt Optimization


In [ ]:
# YOUR CODE
pass




<details><summary>Solution</summary>


In [ ]:
# Token estimator (~4 chars/token, matches tiktoken cl100k_base avg)
# In production: pip install tiktoken; enc = tiktoken.get_encoding('cl100k_base')
def ct(text):
    return max(1, len(text.split()) * 13 // 10)  # ~1.3 tokens/word

bloated = """You are a very helpful, friendly, and knowledgeable
customer support assistant working for Netsetos, which is an
educational technology company headquartered in Hyderabad, India.
Our company specializes in providing high-quality courses in
Artificial Intelligence, Machine Learning, and Generative AI.

When a customer asks a question, provide a helpful and accurate
response. Our refund policy: full refunds within 7 days. After
7 days, 50% refund up to 30 days. After 30 days, no refunds.

Course catalog:
- Complete Generative AI Engineering (Rs 4,999)
- Python for Data Science (Rs 2,999)
- Advanced Machine Learning (Rs 6,999)
- Cloud AI with GCP (Rs 3,999)

Guidelines: Never share customer PII. Verify identity before
account info. Escalate angry customers. Respond in user's
language. Be concise. Include links when available.
Hours: Mon-Sat 9-6 IST. Email: support@netsetos.com.
Always ask if anything else is needed."""

compressed = """Netsetos support (edtech, Hyderabad). AI/ML/GenAI courses.
Refund: 7d=100%, 8-30d=50%, 30d+=none.
Courses: GenAI Rs4999, Python Rs2999, ML Rs6999, GCP Rs3999.
Rules: no PII. Verify ID. Escalate anger. Match language. Be concise.
Hours: Mon-Sat 9-6 IST. support@netsetos.com. End: 'Anything else?'"""

orig = ct(bloated)
comp = ct(compressed)
red = (1 - comp / orig) * 100

print("System Prompt Compression:")
print(f"  Original:   {orig} tokens")
print(f"  Compressed: {comp} tokens")
print(f"  Reduction:  {red:.0f}%")

print(f"\nMonthly Savings (GPT-4o $2.50/MTok):")
for label, vol in [("100K", 100_000), ("500K", 500_000), ("1M", 1_000_000)]:
    saved = (orig - comp) * vol * 2.50 / 1e6
    print(f"  {label}: ${saved:>8.2f} (Rs {saved*84:>8,.0f})")




</details>


---
## Exercise 5 (Medium): Output Format Cost Comparison


In [ ]:
# YOUR CODE
pass




<details><summary>Solution</summary>


In [ ]:
# Token estimator (~4 chars/token, matches tiktoken cl100k_base avg)
# In production: pip install tiktoken; enc = tiktoken.get_encoding('cl100k_base')
def ct(text):
    return max(1, len(text.split()) * 13 // 10)  # ~1.3 tokens/word

prose = """Here are the top 5 products: 1. Complete GenAI Engineering
course priced at Rs 4,999 with 4.8 star rating. 2. Python for Data
Science at Rs 2,999 with 4.6 stars. 3. Advanced ML at Rs 6,999
with 4.9 stars. 4. Cloud AI GCP at Rs 3,999 with 4.5 stars.
5. Django Web Dev at Rs 1,999 with 4.7 stars."""

json_fmt = '[{"name":"GenAI","price":4999,"rating":4.8},{"name":"Python","price":2999,"rating":4.6},{"name":"ML","price":6999,"rating":4.9},{"name":"GCP","price":3999,"rating":4.5},{"name":"Django","price":1999,"rating":4.7}]'

tsv_fmt = "name\tprice\trating\nGenAI\t4999\t4.8\nPython\t2999\t4.6\nML\t6999\t4.9\nGCP\t3999\t4.5\nDjango\t1999\t4.7"

telegraphic = "GenAI 4999 4.8|Python 2999 4.6|ML 6999 4.9|GCP 3999 4.5|Django 1999 4.7"

formats = [("Prose", prose), ("JSON", json_fmt),
           ("TSV", tsv_fmt), ("Telegraphic", telegraphic)]

prose_tok = ct(prose)
print(f"{'Format':<15} {'Tokens':<10} {'vs Prose':<12} {'$/1M req (GPT-4o out)'}")
print("=" * 55)
for name, text in formats:
    tok = ct(text)
    vs = (1 - tok / prose_tok) * 100
    cost = tok * 10.00
    print(f"{name:<15} {tok:<10} {vs:>+6.0f}%       ${cost:>8.2f}")

print(f"\nAt 1M req/month (output $10/MTok):")
for name, text in formats:
    tok = ct(text)
    monthly = tok * 10.00
    print(f"  {name:<15} ${monthly:>6.2f} (Rs {monthly*84:>6,.0f})")

print(f"\nJSON = best for APIs (parseable + compact)")
print(f"TSV = best for tabular data")
print(f"Telegraphic = best for internal pipelines")




</details>


---
## Exercise 6 (Challenge): Tiered Routing System


In [ ]:
# YOUR CODE
pass




<details><summary>Solution</summary>


In [ ]:
import re

TIERS = {
    "budget":   {"model": "deepseek-v3", "input": 0.27,
                 "output": 1.10, "success": 0.85},
    "mid":      {"model": "gpt-4o-mini", "input": 0.15,
                 "output": 0.60, "success": 0.92},
    "premium":  {"model": "claude-sonnet", "input": 3.00,
                 "output": 15.00, "success": 0.97},
    "frontier": {"model": "o3", "input": 10.00,
                 "output": 40.00, "success": 0.99},
}

def classify(q):
    ql = q.lower()
    for kw in ["prove", "theorem", "novel research", "invent"]:
        if kw in ql: return "frontier"
    for kw in ["design.*arch", "build.*pipeline", "code review", "multi-step"]:
        if re.search(kw, ql): return "premium"
    for kw in ["explain", "compare", "write", "summarize", "debug", "create"]:
        if kw in ql: return "mid"
    return "budget"

queries = [
    "What is Python?", "Hello!", "Translate to Hindi",
    "Define REST API", "What time is it?",
    "Explain microservices vs monolith",
    "Compare React and Vue", "Write unit tests",
    "Summarize this email", "Create a tagline",
    "Debug null pointer", "Draft a proposal",
    "Explain OAuth 2.0", "Write churn analysis SQL",
    "Compare AWS vs GCP pricing",
    "Design a microservices architecture for payments",
    "Build a RAG pipeline with hybrid search",
    "Code review authentication module",
    "Multi-step reasoning: optimize supply chain",
    "Design architecture for 10M users",
    "Prove algorithm is O(n log n)",
    "Novel research: new attention mechanism",
    "What is 2+2?", "List 5 fruits",
    "Write a haiku", "Create REST endpoint",
    "Explain transformers", "Debug this function",
    "Hello world in Go", "Compare databases",
]

avg_in, avg_out = 200, 150
results = {"budget": 0, "mid": 0, "premium": 0, "frontier": 0}
total_cost = total_cpso = 0

for i, q in enumerate(queries, 1):
    tier = classify(q)
    results[tier] += 1
    t = TIERS[tier]
    cost = (avg_in * t["input"] + avg_out * t["output"]) / 1e6
    cpso = cost / t["success"]
    total_cost += cost
    total_cpso += cpso

baseline = len(queries) * (avg_in * 3.00 + avg_out * 15.00) / 1e6
savings = (1 - total_cost / baseline) * 100

print(f"Distribution: {results}")
print(f"Baseline (all Sonnet): ${baseline:.6f}")
print(f"Tiered:                ${total_cost:.6f} (-{savings:.0f}%)")
print(f"Avg CPSO:              ${total_cpso/len(queries):.6f}")

print(f"\nCPSO per Tier:")
for tier in TIERS:
    if results[tier] == 0: continue
    t = TIERS[tier]
    c = (avg_in*t["input"] + avg_out*t["output"]) / 1e6
    print(f"  {tier:<10} ${c/t['success']:.6f} (success: {t['success']*100:.0f}%)")

m_base = baseline / len(queries) * 300_000
m_tier = total_cost / len(queries) * 300_000
print(f"\nMonthly (10K/day):")
print(f"  Baseline: Rs {m_base*84:>10,.0f}")
print(f"  Tiered:   Rs {m_tier*84:>10,.0f}")
print(f"  Savings:  Rs {(m_base-m_tier)*84:>10,.0f}")




</details>


---
## Exercise 7 (Challenge): India Cost Calculator


In [ ]:
# YOUR CODE
pass




<details><summary>Solution</summary>


In [ ]:
def india_cost(model, words, lang="english", region="us"):
    indic = {"english": 1.16, "hindi": 1.89, "tamil": 2.32,
             "telugu": 2.10, "bengali": 2.05}
    tpw = indic.get(lang, 1.5)
    tokens = words * tpw
    prices = {"gpt-4o": 6.25, "gpt-4o-mini": 0.375,
              "claude-sonnet": 9.00, "gemini-flash": 0.1875,
              "deepseek-v3": 0.685, "sarvam-105b": 0.00}
    base = tokens * prices.get(model, 1.0) / 1e6
    gst = 0.18 if region == "us" else 0
    forex = 0.025 if region == "us" else 0
    total_usd = base * (1 + gst + forex)
    total_inr = total_usd * 84
    penalty = (tpw / 1.16 - 1) * 100
    return {"model": model, "tokens": tokens, "base": base,
            "gst": gst*100, "penalty": penalty,
            "total_usd": total_usd, "total_inr": total_inr}

words = 10_000_000
models = [("gpt-4o", "us"), ("gpt-4o-mini", "us"),
          ("gemini-flash", "india"), ("deepseek-v3", "us"),
          ("sarvam-105b", "india")]

for lang in ["english", "hindi"]:
    print(f"\n--- {lang.upper()} ({words/1e6:.0f}M words/mo) ---")
    print(f"{'Model':<16} {'Tokens':<10} {'Base $':<9} "
          f"{'Indic %':<8} {'Total INR'}")
    print("-" * 60)
    for model, region in models:
        r = india_cost(model, words, lang, region)
        print(f"{r['model']:<16} {r['tokens']/1e6:>6.1f}M "
              f"${r['base']:>7.2f} {r['penalty']:>+5.0f}%  "
              f"Rs {r['total_inr']:>10,.0f}")

print("\nHindi costs 63% more tokens than English on GPT-4o")
print("Sarvam AI: free LLMs, eliminates Indic penalty")




</details>


---
## Exercise 8 (Challenge): Full Cost Optimization Stack


In [ ]:
# YOUR CODE
pass




<details><summary>Solution</summary>


In [ ]:
baseline = 500.0

layers = [
    ("1. System Prompt Compression", 0.15),
    ("2. Output Format (JSON)", 0.10),
    ("3. Semantic Caching (55% hit)", 0.45),
    ("4. Tiered Routing (70% Flash)", 0.60),
    ("5. Batch API (30% eligible)", 0.20),
]

print(f"Baseline: ${baseline:.0f}/month (Rs {baseline*84:,.0f})")
print("=" * 60)
current = baseline
print(f"{'Layer':<40} {'Red %':<8} {'After $':<10} {'Cumul %'}")
print("-" * 65)
for name, pct in layers:
    current -= current * pct
    cumul = (1 - current / baseline) * 100
    print(f"{name:<40} -{pct*100:.0f}%    "
          f"${current:>7.2f}   {cumul:.0f}%")

print(f"\nFinal: ${current:.2f}/month (Rs {current*84:,.0f})")
print(f"Saved: ${baseline-current:.2f} ({(1-current/baseline)*100:.0f}%)")
print(f"Target <$100: {'ACHIEVED' if current < 100 else 'NOT MET'}")

print(f"\nLayer Contributions:")
c2 = baseline
for name, pct in layers:
    saved = c2 * pct
    print(f"  {name:<40} ${saved:>6.2f} ({saved/baseline*100:.1f}%)")
    c2 -= saved

print(f"\nCPSO Warning: cheapest per-token != cheapest per-task")
print(f"Gitar: 5x cheaper model + 2.5x retries = MORE expensive")




</details>


---
## Exercise 3 (Medium): Helicone Cost Dashboard
Keyless local simulation: build a `CostDashboard` with graduated budget enforcement (alert/throttle/downgrade/block), simulate 100 tagged requests, and report cost by model/feature/user.

In [ ]:
# YOUR CODE
pass

<details><summary>Solution</summary>

In [ ]:
import random, json
from collections import defaultdict
from datetime import datetime

class CostDashboard:
    def __init__(self, daily_budget=10.0):
        self.budget = daily_budget
        self.records = []
        self.alerts = []

    def log(self, model, feature, user_type,
            in_tok, out_tok, cost, latency_ms):
        self.records.append({
            "ts": datetime.now().isoformat(),
            "model": model, "feature": feature,
            "user_type": user_type,
            "in_tok": in_tok, "out_tok": out_tok,
            "cost": cost, "latency_ms": latency_ms,
        })
        # Graduated enforcement
        spent = sum(r["cost"] for r in self.records)
        pct = spent / self.budget * 100
        if pct >= 100:
            self.alerts.append(("BLOCK", spent))
        elif pct >= 90:
            self.alerts.append(("DOWNGRADE", spent))
        elif pct >= 80:
            self.alerts.append(("THROTTLE", spent))
        elif pct >= 50:
            self.alerts.append(("ALERT", spent))

    def report(self):
        by_model = defaultdict(lambda: {"cost": 0, "n": 0})
        by_feature = defaultdict(lambda: {"cost": 0, "n": 0})
        by_user = defaultdict(lambda: {"cost": 0, "n": 0})

        for r in self.records:
            by_model[r["model"]]["cost"] += r["cost"]
            by_model[r["model"]]["n"] += 1
            by_feature[r["feature"]]["cost"] += r["cost"]
            by_feature[r["feature"]]["n"] += 1
            by_user[r["user_type"]]["cost"] += r["cost"]
            by_user[r["user_type"]]["n"] += 1

        total = sum(r["cost"] for r in self.records)
        print("Cost Dashboard Report:")
        print("=" * 55)

        print(f"\nBy Model:")
        for m, d in sorted(by_model.items(),
                           key=lambda x: -x[1]["cost"]):
            pct = d["cost"]/total*100 if total else 0
            print(f"  {m:<20} ${d['cost']:.4f} "
                  f"({d['n']} calls, {pct:.0f}%)")

        print(f"\nBy Feature:")
        for f, d in sorted(by_feature.items(),
                           key=lambda x: -x[1]["cost"]):
            print(f"  {f:<20} ${d['cost']:.4f} "
                  f"({d['n']} calls)")

        print(f"\nBy User Type:")
        for u, d in sorted(by_user.items(),
                           key=lambda x: -x[1]["cost"]):
            print(f"  {u:<20} ${d['cost']:.4f} "
                  f"({d['n']} calls)")

        print(f"\nTotal: ${total:.4f} / "
              f"${self.budget:.2f} budget "
              f"({total/self.budget*100:.0f}%)")
        if self.alerts:
            last = self.alerts[-1]
            print(f"Last alert: {last[0]} at ${last[1]:.4f}")

# Simulate 100 requests
dash = CostDashboard(daily_budget=0.50)
models = ["gpt-4o", "gpt-4o-mini", "gemini-flash"]
features = ["chat", "search", "summarize", "classify"]
users = ["free", "pro", "enterprise"]

for _ in range(100):
    model = random.choice(models)
    cost_map = {"gpt-4o": 0.005, "gpt-4o-mini": 0.0003,
                "gemini-flash": 0.0001}
    cost = cost_map[model] * (0.5 + random.random())
    dash.log(model, random.choice(features),
             random.choice(users),
             random.randint(50, 500),
             random.randint(20, 200),
             cost, random.randint(100, 3000))

dash.report()

print("\n8 Dashboard Metrics:")
for i, m in enumerate([
    "Cost per request (avg + p95)",
    "Cost per user (attribution)",
    "Cost per feature (chat vs search)",
    "Cost per conversation (multi-turn)",
    "Cache hit ratio (%)",
    "Model distribution (% per model)",
    "Cost trend (daily/weekly)",
    "Error/retry rate (%)",
], 1):
    print(f"  {i}. {m}")

print("\nTool Comparison:")
tools = [
    ("Helicone", "1-line URL", "10K free", "Fastest deploy"),
    ("Langfuse", "SDK decorator", "50K free", "Agent tracing"),
    ("LiteLLM", "Proxy server", "Free OSS", "Budget enforce"),
]
for name, integ, free, best in tools:
    print(f"  {name:<10} {integ:<14} {free:<10} {best}")

</details>

---
## Exercise 4 (Medium): Prompt Caching Implementation
Keyless arithmetic simulation: compute caching economics for a 5K-token system prompt across Anthropic/OpenAI/Google at 3 volume levels, stack cache + batch savings, and find the break-even point.

In [ ]:
# YOUR CODE
pass

<details><summary>Solution</summary>

In [ ]:
cached_tokens = 5000
user_tokens = 300
output_tokens = 200

providers = [
    {"name": "Anthropic (Sonnet)",
     "input": 3.00, "output": 15.00,
     "cache_write": 1.25, "cache_read": 0.10,
     "batch_discount": 0.50,
     "min_tokens": 1024, "ttl": "5 min"},
    {"name": "OpenAI (GPT-4o)",
     "input": 2.50, "output": 10.00,
     "cache_write": 1.00, "cache_read": 0.50,
     "batch_discount": 0.50,
     "min_tokens": 1024, "ttl": "5-10 min (auto)"},
    {"name": "Google (Gemini 2.5)",
     "input": 1.25, "output": 5.00,
     "cache_write": 1.00, "cache_read": 0.10,
     "batch_discount": 0.50,
     "min_tokens": 32768, "ttl": "configurable"},
]

volumes = [10_000, 100_000, 1_000_000]

print("Prompt Caching Economics:")
print(f"System prompt: {cached_tokens} tokens")
print(f"User input: {user_tokens} tokens")
print(f"Output: {output_tokens} tokens")
print("=" * 75)

for p in providers:
    print(f"\n{p['name']}:")
    total_in = cached_tokens + user_tokens
    for vol in volumes:
        # No cache
        no_cache = vol * (
            total_in * p["input"]
            + output_tokens * p["output"]
        ) / 1e6

        # With cache
        write = 1 * cached_tokens * p["input"] * p["cache_write"] / 1e6
        reads = (vol-1) * cached_tokens * p["input"] * p["cache_read"] / 1e6
        fresh = vol * user_tokens * p["input"] / 1e6
        out = vol * output_tokens * p["output"] / 1e6
        with_cache = write + reads + fresh + out

        # Cache + batch
        cache_batch = with_cache * p["batch_discount"]

        save_c = (1 - with_cache/no_cache) * 100
        save_cb = (1 - cache_batch/no_cache) * 100

        print(f"  {vol:>10,} calls: "
              f"no_cache=${no_cache:>8.2f}  "
              f"cache=${with_cache:>8.2f} (-{save_c:.0f}%)  "
              f"cache+batch=${cache_batch:>8.2f} (-{save_cb:.0f}%)")

    # Break-even: when does cache write cost pay off?
    write_cost = cached_tokens * p["input"] * p["cache_write"] / 1e6
    saving_per_read = cached_tokens * p["input"] * (1 - p["cache_read"]) / 1e6
    breakeven = write_cost / saving_per_read if saving_per_read > 0 else float('inf')
    print(f"  Break-even: {breakeven:.1f} calls "
          f"(write cost: ${write_cost:.6f})")

print(f"\nKey insight: Anthropic cache+batch = 95% savings")
print(f"$3.00/M -> $0.15/M input tokens")

</details>